# 02 — LightGBM smoking classifier (PyCaret)

Original PyCaret LightGBM smoking classifier from `220914/predict/smk-checkpoint.ipynb`.
Produces the AUC=0.73 ROC curve and SHAP feature-importance panel of **Supplementary Figure 3** of Bao et al. (NTR-2026-092).

## Inputs

- `data/predict_smk.txt` — the smoker-classifier feature matrix (tab-separated). 7,009 rows x 72 columns.
  - 71 OTU relative-abundance feature columns (selected per the manuscript Methods) plus the
    `Districts` target column.
  - The notebook target name `Districts` is a legacy column header from an earlier project draft;
    in the published paper this column encodes binary smoker (`1`) vs never-smoker (`0`) status,
    so the classifier is binary smoking-status. **Do not** confuse with geographic district.
  - This input file is NOT redistributed; it is regenerable from `data/GGMP7009_even10k.biom` +
    `data/GPf_metadata.tsv` + `data/smk_status_sig_res.tsv` by the same OTU-selection logic
    documented in the manuscript Methods (also see `scripts/revision/04_lightgbm_optimized.py`
    which performs the analogous variant screen with Optuna).

## Outputs

- ROC / AUC plots and SHAP panel rendered inline in the notebook (used to assemble Supp Fig 3).
- No on-disk outputs — the original analysis was run interactively and figures were exported via PyCaret's `plot_model` / `interpret_model` UI.

## Environment

- Python 3.8 with PyCaret 2.x (per the original `~/opt/anaconda3/envs/pycaret` environment),
  `lightgbm`, `pandas`, `shap`. PyCaret 2.x is required because the API in PyCaret 3.x diverges.

## How to run

Open this notebook in JupyterLab / Jupyter Notebook with the PyCaret 2.x kernel selected,
place `predict_smk.txt` at `data/predict_smk.txt`, and execute cells top-to-bottom.

Reproducibility seeds:

- `dataset.sample(frac=0.9, random_state=786)` — fixed train/holdout split.
- `setup(..., session_id=123)` — fixed PyCaret session (StratifiedKFold seed 123).
- `tune_model(..., n_iter=20)` — 20-iter random search over LightGBM hyperparameters.

## Note on cells 9-10 (`blend_models` / `stack_models`)

These cells are kept as in the original notebook for fidelity. Cell 9 (`blend_models`) was a
user error in the original session — `blend_models` expects a list, not a single estimator,
so it raised `TypeError`. Skip cell 9; the AUC-0.73 result reported in the manuscript comes
from `tuned_lightgbm` produced by cell 6.


In [ ]:
import pandas as pd

In [ ]:
# Load the smoker-classifier feature matrix.
# Default path is `data/predict_smk.txt` relative to the repository root.
dataset = pd.read_table("data/predict_smk.txt", sep="\t")
data = dataset.sample(frac=0.9, random_state=786).reset_index(drop=True)
data_unseen = dataset.drop(data.index).reset_index(drop=True)

print('Data for Modeling: ' + str(data.shape))
print('Unseen Data For Predictions: ' + str(data_unseen.shape))

In [ ]:
from pycaret.classification import *

In [ ]:
# `target='Districts'` is the legacy column name encoding binary smoker vs never-smoker
# (see notebook header). Do not change without updating `predict_smk.txt`.
exp_mclf101 = setup(data=data, target='Districts', session_id=123, feature_selection=True, n_jobs=1)

In [ ]:
best_model = compare_models()

In [ ]:
lightgbm = create_model('lightgbm')

In [ ]:
# 20-iter random search; produces the published AUC=0.73 LightGBM model used in Supp Fig 3.
tuned_lightgbm = tune_model(lightgbm, n_iter=20)

In [ ]:
rf = create_model('rf')

In [ ]:
tuned_rf = tune_model(rf, n_iter=20)

**Note:** the next cell (`blend_models(tuned_lightgbm)`) errored in the original notebook
because `blend_models` expects a list of estimators, not a single one. Kept here for fidelity
with the original; skip in re-runs.


In [ ]:
# blender = blend_models(tuned_lightgbm)  # original error; see markdown note above

In [ ]:
stacker = stack_models(tuned_rf)

## Plot ROC / AUC and SHAP panels (Supp Fig 3)

Use PyCaret's interactive `plot_model` / `interpret_model` calls to render the ROC curve
(AUC ≈ 0.73) and SHAP feature-importance panels reproduced in Supp Fig 3.


In [ ]:
# ROC curve for the tuned LightGBM model (Supp Fig 3 left panel).
plot_model(tuned_lightgbm, plot='auc')

In [ ]:
# SHAP feature-importance panel (Supp Fig 3 right panel).
interpret_model(tuned_lightgbm)